# Understanding NIRCam subarrays and Apertures

The [JDox page on NIRCam subarrays](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-instrumentation/nircam-detector-overview/nircam-detector-subarrays#gsc.tab=0)
explains the subarrays pretty well.
However, I wanted to have a notebook to quickly find and explore datasets for specific subarrays.
More specifically, I wanted to query images with the `FULLP` subarray.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["image.origin"] = "lower"

data_path = Path("./data/")

## Queries based on observation info

In [ ]:
from astroquery.mast import Observations

meta_tbl = Observations.get_metadata("observations")
print(meta_tbl)

For JWST queries, the following should return all NIRCam science imaging stage 2 products for filters `F480M` and `F430M`.

In [ ]:
obs_tbl = Observations.query_criteria(
    intentType="science",
    obs_collection="JWST",
    filters=["F480M", "F430M"],
    instrument_name="NIRCAM/IMAGE",
    calib_level=2,
    dataRights="PUBLIC",
)

If we don't restrict the program ID with `proposal_id`, we will have a lot of observations in the table.
From it we can get a products_list.

In [ ]:
product_tbl = Observations.get_product_list(obs_tbl)
product_tbl = Observations.filter_products(
    product_tbl, productType="SCIENCE", calib_level=[2], extension=["fits"], productSubGroupDescription=["CAL"], filters=["F480M", "F430M"]
)

Again, if we don't filter based on proposal, this table will be very large.
When filtering based on proposal, this will return a fairly short and useful list.
However, I have not found these queries super-useful to search all data based on a specific observing setup.
My understanding is that the `MastMissions` interface is better suited for that.

## Querying based on observing setup

We can create a mission object for JWST and list its columns.

In [ ]:
from astroquery.mast import MastMissions

mission = MastMissions(mission="jwst")

In [ ]:
mission.get_column_list()

In [ ]:
mission_tbl = mission.query_criteria(instrume="NIRCAM", access="PUBLIC", subarray="FULLP")
default_mission_columns = mission_tbl.colnames

The issue here is that the `FULLP` subarray is not reflected in the `subarray` keyword.
We will need to filter it based on other keywords.
To figure this out, I searched online for a program using `FULLP` to inspect one of their data products.

## Finding images for a given program

We will download data from program 3772, which uses the `FULLP` subarray, according to [this paper](https://iopscience.iop.org/article/10.3847/1538-4357/ad21fa).

In [ ]:
program = 3772
obs_tbl = Observations.query_criteria(
    proposal_id=program,
    obs_collection="JWST",
    instrument_name="NIRCAM/IMAGE",
)
products_tbl = Observations.get_product_list(obs_tbl)
products_tbl = Observations.filter_products(
    products_tbl, productType="SCIENCE", calib_level=[2], extension=["fits"], productSubGroupDescription=["CAL"],
)
products_tbl.sort("obs_id")
products_tbl

As expected, only `nrcb1` was used in the SW channel.
We can filter for the F360M observations to get a data product in the LW channel and inspect its content.

In [ ]:
products_tbl = Observations.filter_products(products_tbl, filters="F360M")

In [ ]:
uri = products_tbl["dataURI"][0]

path = data_path / uri.split("/")[-1]
Observations.download_file(uri, local_path=path);

In [ ]:
from astropy.io import fits

hdul = fits.open(path)
hdr = hdul[0].header
img = hdul["SCI"].data
sci_hdr = hdul["SCI"].header

Let us first look at some info from the header.

In [ ]:
print("SUBARRAY:", hdr["SUBARRAY"])
print("APERNAME:", hdr["APERNAME"])
print("PPS_APER:", hdr["PPS_APER"])
print("XOFFSET:", hdr["XOFFSET"])
print("YOFFSET:", hdr["YOFFSET"])

As we can see, the FULLP subarray is not reflected in the `SUBARRAY` keyword.
Instead, it is reflected in the `PPS_APER` and `APERNAME` keywords.
This is good to know if we want to do a data search based on header keys later on.

Also note that the offset is basically 0, meaning that the target should be near the pointing reference position.
This pointing reference position is given by the aperture.
Unfortunately, the FULLP subarray is not mentioned in the [JDox page on apertures](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-operations/nircam-apertures#gsc.tab=0).
My understanding is that the "zero" position is not that of B5, nor of B1, but rather closer to the top right corner.
The dataset we have at hand is not great to test this since it targets a faint source.

We can still overplot:

- The CRPIX reference pixels
- The V2 and V3 reference pixels from the header
- The V2 and V3 reference pixels from JDox for the B1 aperture.

In [ ]:
from coldest import pointing

xref, yref = sci_hdr["CRPIX1"], sci_hdr["CRPIX2"]
v2_ref, v3_ref = sci_hdr["V2_REF"], sci_hdr["V3_REF"]
xref_v2v3, yref_v2v3 = pointing.v2v3_to_xy(v2_ref, v3_ref, path)
v2_ref_jdox = -122.43
v3_ref_jdox = -457.71
xref_jdox_v2v3, yref_jdox_v2v3 = pointing.v2v3_to_xy(v2_ref_jdox, v3_ref_jdox, path)
print("X ref: ", xref)
print("Y ref: ", xref)
print("V2 ref: ", v2_ref)
print("V3 ref: ", v3_ref)

In [ ]:
plt.imshow(img, norm="symlog")
plt.plot(xref, yref, "r+", label="Ref pix position")
plt.plot(xref_v2v3, yref_v2v3, "yx", label="V2-V3 Ref position")
plt.plot(xref_jdox_v2v3, yref_jdox_v2v3, "y+", label="V2-V3 Ref position B1 aperture")
plt.colorbar()
plt.show()

I won't do a detailed crop here, but already from this it looks like the reference positions from the header are close to a field that looks like the figure in the paper linked above.
In any case, we can now look at other datasets that use the `FULLP` subarray.

## Querying all data with the FULLP aperture

To query data with a specific observing setup, we will use the `MastMissions` interface with some extra columns in the output.

In [ ]:
select_col = default_mission_columns + ["pps_aper", "xoffset", "yoffset", "filter", "title"]
mission_tbl = mission.query_criteria(instrume="NIRCAM", access="PUBLIC", pps_aper="NRCB5_FULLP", exp_type="NRC_IMAGE", select_cols=select_col, filter=["F480M", "F430M"])

In [ ]:
products_tbl = mission.get_unique_product_list(mission_tbl)

In [ ]:
products_tbl = mission.filter_products(
    products_tbl, type="science", file_suffix=["_cal"]
)

In [ ]:
products_tbl = products_tbl[["nrcblong" in f for f in products_tbl["filename"]]]

In [ ]:
products_tbl

Let us just pick the first data product from that list and look at it.

In [ ]:
product_row = products_tbl[0]
path = data_path / product_row["filename"]
mission.download_file(product_row["uri"], local_path=path)

In [ ]:
hdul = fits.open(path)
hdr = hdul[0].header
img = hdul["SCI"].data
sci_hdr = hdul["SCI"].header

In [ ]:
print("SUBARRAY:", hdr["SUBARRAY"])
print("APERNAME:", hdr["APERNAME"])
print("PPS_APER:", hdr["PPS_APER"])
print("XOFFSET:", hdr["XOFFSET"])
print("YOFFSET:", hdr["YOFFSET"])

xref, yref = sci_hdr["CRPIX1"], sci_hdr["CRPIX2"]
v2_ref, v3_ref = sci_hdr["V2_REF"], sci_hdr["V3_REF"]

print("X ref: ", xref)
print("Y ref: ", xref)
print("V2 ref: ", v2_ref)
print("V3 ref: ", v3_ref)

In [ ]:
xref_v2v3, yref_v2v3 = pointing.v2v3_to_xy(v2_ref, v3_ref, path)
xref_jdox_v2v3, yref_jdox_v2v3 = pointing.v2v3_to_xy(v2_ref_jdox, v3_ref_jdox, path)

xoffset = hdr["XOFFSET"]
yoffset = hdr["YOFFSET"]

x_target, y_target = pointing.apply_pointing(xoffset, yoffset, path)

In [ ]:
plt.imshow(img, norm="symlog")
plt.plot(xref, yref, "r+", label="Ref pix position")
plt.plot(xref_v2v3, yref_v2v3, "yx", label="V2-V3 Ref position")
plt.plot(xref_jdox_v2v3, yref_jdox_v2v3, "y+", label="V2-V3 Ref position B1 aperture")
plt.plot(x_target, y_target, "r*", label="Expected target pointing")
plt.colorbar()
plt.show()

Again, based on this, it seems the pointing is taken from the reference position in the headers, which would be: `v2=-133.18` and `v3=-446.80`.
I was able to confirm this via a pointing file exported from the APT for program 3772.

## Searching data with the SUB400P subarray

In [ ]:
mission_tbl = mission.query_criteria(instrume="NIRCAM", access="PUBLIC", subarray="SUB400P", exp_type="NRC_IMAGE", select_cols=select_col, filter=["F480M", "F430M"])
print(np.unique(mission_tbl["title"]))

In [ ]:
# title = "Confirmation of a Jovian Planet Analog Orbiting a White Dwarf, Rare Low-mass Neutron Star or Black Hole"
title = "Formation Histories and Stellar Masses of Very High-z Quasars"
mission_tbl = mission_tbl[mission_tbl["title"] == title]

In [ ]:
products_tbl = mission.get_unique_product_list(mission_tbl)
products_tbl = mission.filter_products(
    products_tbl, type="science", file_suffix=["_cal"],
)
products_tbl = products_tbl[["nrcblong" in f for f in products_tbl["filename"]]]

In [ ]:
product_row = products_tbl[0]
path = data_path / product_row["filename"]
mission.download_file(product_row["uri"], local_path=path)

In [ ]:
hdul = fits.open(path)
hdr = hdul[0].header
img = hdul["SCI"].data
sci_hdr = hdul["SCI"].header

In [ ]:
print("SUBARRAY:", hdr["SUBARRAY"])
print("APERNAME:", hdr["APERNAME"])
print("PPS_APER:", hdr["PPS_APER"])
print("XOFFSET:", hdr["XOFFSET"])
print("YOFFSET:", hdr["YOFFSET"])

xref, yref = sci_hdr["CRPIX1"], sci_hdr["CRPIX2"]
v2_ref, v3_ref = sci_hdr["V2_REF"], sci_hdr["V3_REF"]

print("X ref: ", xref)
print("Y ref: ", xref)
print("V2 ref: ", v2_ref)
print("V3 ref: ", v3_ref)

In [ ]:
xref_v2v3, yref_v2v3 = pointing.v2v3_to_xy(v2_ref, v3_ref, path)
xref_jdox_v2v3, yref_jdox_v2v3 = pointing.v2v3_to_xy(v2_ref_jdox, v3_ref_jdox, path)

xoffset = hdr["XOFFSET"]
yoffset = hdr["YOFFSET"]

x_target, y_target = pointing.apply_pointing(xoffset, yoffset, path)

In [ ]:
plt.imshow(img, norm="symlog")
plt.plot(xref, yref, "r+", label="Ref pix position")
plt.plot(xref_v2v3, yref_v2v3, "yx", label="V2-V3 Ref position")
plt.plot(x_target, y_target, "r*", label="Expected target pointing")
plt.colorbar()
plt.show()